In [ ]:
from sympy import symbols, Eq, solve
import re
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

token = os.environ.get("HUGGING_TOKEN")

In [ ]:
from transformers import pipeline

In [ ]:
generator = pipeline("text-generation", model="mistralai/Ministral-8B-Instruct-2410", token=token, temperature=0.001)

In [42]:
prompt = "If you asked me how much X equals, I'd say... hundred."
instruction = [
    {
        "role": "system", "content":
            "You are a natural language equation parser.\n"
            "1. Output ONLY a comma-separated list of equations. Each equation must:\n"
            "   - Be in single quotes: 'example'\n"
            "   - Consist of two expressions separated by =\n"
            "   - Both of the expressions must consist of named variables, numbers, and operators between them"
            "   - The named variables consist of latin characters only, e.g. x, y, var, john, apple etc.\n"
            "   - The numbers should use '.' for decimal points when necessary\n"
            "   - The only operators allowed are: +, -, *, /, ** and there should be spaces on both sides of each operator \n"
            "2. If the input describes an inequality (>, <, >=, <=, !=, or their verbal forms), respond ONLY with: INEQUAL_WARNING.\n"
            "3. If the input does not describe a valid math equation, respond ONLY with: NOTMATH_WARNING.\n"
            "Do not solve the equations. Do not explain anything. Do not output code. Output nothing except what the rules above require."
    },
    {"role": "user", "content": f"{prompt}\n"},
]


In [43]:
result = generator(instruction)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [44]:
def result_parser (result) :
    answer = result[0]["generated_text"] [2] ['content']

    answers = answer.split(', ')
    answers_cleared = []

    for elem in answers :
        elem_clr = elem.replace("'", "")

        answers_cleared.append(elem_clr)

    return answers_cleared


In [45]:
def explicit_multiplication (equation) :
    import re

    pattern = r'([0-9])\s*([A-Za-z])|([A-Za-z])\s*([0-9])'

    def insert_multiply(match):
        if match.group(1) and match.group(2):
            # digit+letter
            return f"{match.group(1)} * {match.group(2)}"
        else:
            # letter+digit
            return f"{match.group(3)} * {match.group(4)}"

    eq_expl = re.sub(pattern, insert_multiply, equation)
    return eq_expl

In [46]:
parsed_answer = result_parser(result)

equations = []
for eq in parsed_answer :
    equations.append(explicit_multiplication(eq))

print(equations)

['x = 100']


In [47]:
import re

def is_safe_equation(s: str) -> bool:
    if s.count('=') != 1: return False # only one equation sign
    if re.search(r"[\"'`_<>!^&|:%,$\\\[\]{}]", s): return False # forbidden chars
    if not re.fullmatch(r"[A-Za-z0-9+\-*/=().\s]+", s): return False # allowed chars
    if re.search(r"[A-Za-z]\s*\.\s*[A-Za-z0-9]", s): return False # a.b not allowed
    if re.search(r"[A-Za-z][A-Za-z0-9]*\s*\(", s): return False # not allowed fun(
    L, R = (p.strip() for p in s.split('='))
    if not L or not R: return False # something on both sides of equation

    bal = 0
    for ch in s: # both brackets present
        bal += (ch == '(') - (ch == ')')
        if bal < 0: return False
    return bal == 0

In [48]:
for eq in equations :
    if not is_safe_equation(eq) :
        raise Exception(f"The equation {eq} is not safe.")

In [49]:
symbol_names = set()
for eq in equations :
    matches = re.findall(r"[A-Za-z]+", eq)
    symbol_names.update(matches)

print(f'Symbol names: {symbol_names}')

Symbol names: {'x'}


In [50]:
from sympy import symbols

sympy_symbols = symbols(" ".join(symbol_names))

In [51]:
from sympy import sympify
from sympy import Eq

equation_set = []

for eq in equations :
    left, right = eq.split('=')

    equation_set.append(
        Eq(sympify(left), sympify(right))
    )

In [52]:
solve(equation_set, sympy_symbols, dict=True)

[{x: 100}]